# W06S01 — Open the GOVCOPP case and describe coauthorship from authorship links

**Goal of the practice block.** Starting from the two GOVCOPP tables, audit the authorship
links, build the bipartite authorship network, project it onto researchers and read the
coauthorship network it produces, including how far its ties cross the formal groups.

**Project question.** To what extent does the formal organisational structure of the GOVCOPP
research groups correspond to the thematic structure observable in their researchers'
scientific production?

**Today's question.** How is observed coauthorship among GOVCOPP researchers structured, and to
what extent do its ties cross the formal groups?

Today describes *who signs publications with whom*. Themes come in W07–W09: coauthorship does
not show that two researchers work on the same topic, and it does not validate the formal groups.

**Learning outcomes:** OA1.1, OA1.2, OA2.1, OA2.2, OA2.3, OA4.1, OA4.2, OA9.1.

| Section | When | Minutes | Evidence it produces |
|---|---|---:|---|
| 0–3 Fictitious example | exposition and demo | 20–85 | hand counts confirmed by code |
| 4 Load and audit (checkpoint C1) | groups | 95–120 | audit of keys, pairs and groups |
| 5 Bipartite authorship (C2) | groups | 95–120 | incidence matrix with all links |
| 6 Projection (C3) | groups | 120–145 | weighted network, every researcher kept |
| 7 Measures and figures | groups | 120–145 | degree, strength, components, figures |
| 8 Coauthorship and formal groups (C4) | groups | 145–170 | two shares with units and denominators |
| 9 Group synthesis | groups | 145–180 | 180–250 words, decision log, outputs |
| 10 Optional extension | if you finish | — | sensitivity to a minimum edge weight of 2 |
| After class | home extension | about 1.5 h per student | Research Plan v0, provisional variant, individual run, evidence entry E06 |

**How to work.** Groups of three or four, with rotating roles: *operator* (runs the notebook),
*data checker* (compares every result with the checkpoint), *interpreter* (writes the answers).
Run the cells in order. Run each checkpoint before moving on: if it fails, fix the cell above it.
Finish with **Kernel → Restart & Run All**.

**Packages.** pandas, numpy, scipy, matplotlib and networkx. Before the class, add `scipy` and
`networkx` to your project's `environment.yml` and update the environment (see the W06 README).

| You may change | You must check | You must keep |
|---|---|---|
| the cells marked `YOUR CODE HERE` | every `assert` checkpoint | the file names in `data/raw/` |
| the Markdown answer cells | the units in each table title | the parameters cell |
| the optional extension (section 10) | Restart & Run All at the end | the order of the sections |
**Project adaptation (2026-10-19).** This copy uses the two local CSV files mapped by the project owner. Their group assignments differ from the teacher solution: one researcher belongs to PI3 rather than PI2. Input hashes and group checkpoints below refer to these local files. The original teacher notebook remains in `notebooks/aulas/`.


In [ ]:
from pathlib import Path

here = Path.cwd().resolve()
for folder in (here, *here.parents):
    if (folder / "environment.yml").is_file():
        PROJECT_ROOT = folder
        break
else:
    raise FileNotFoundError(f"environment.yml not found in {here} or above")

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "w06"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


**Where this notebook lives: the W05 project rules.** This is the first notebook of the GOVCOPP
project, and it runs inside the project folder your group built in W05, not in a loose folder.

| W05 rule | What it means in this notebook |
|---|---|
| The project, not the notebook, is the unit of work | Save it as `notebooks/W06S01_practice.ipynb` in your group's repository |
| Build paths from the project root | The first cell finds the folder that holds `environment.yml`; never type `C:/…` and never use `os.chdir` |
| Raw data are read, never edited | The two CSV files go to `data/raw/` exactly as received; the fingerprint check in section 4 proves it |
| Outputs are regenerated, never retouched | Run All rewrites everything in `outputs/w06/coauthorship/`; never edit a figure or a CSV by hand |
| Restart Kernel → Run All is the workflow checkpoint | A result you only saw before a restart does not count |
| Repeated code becomes one function in `src/icd_project/` | Section 4 reads both tables with the same rule: see *After class* |
| AI suggestions are verified before you accept them | Record each one in the AI use log of section 9 |
| A verified change is recorded in a small commit that says why | One commit with the hand-in at the end of the session |

**Inputs:** `data/raw/publications_2020_2026.csv` and
`data/raw/authorship_2020_2026.csv`.
**Decision of the day:** the network keeps every researcher, isolates included, and weights an
edge by the number of distinct publications two researchers share.
**Outputs:** `outputs/w06/coauthorship/` (CSV tables, two figures and `statistics.json`).

## 0 Setup and contract of the analysis

The first cell (above) found your project folder and defined `DATA_DIR` (`data/raw`) and
`OUTPUT_DIR` (`outputs/w06`). The next cell imports the libraries and prints their versions, so
that your results record the environment that produced them.

In [ ]:
import hashlib
import itertools
import json
import math
from collections import Counter
from importlib.metadata import version

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp

VERSIONS = {name: version(name) for name in ("pandas", "numpy", "scipy", "matplotlib", "networkx")}
VERSIONS

**Parameters.** One cell with every choice the analysis depends on. Do not edit it during the
class: the checkpoints assume these values.

In [ ]:
PUBLICATIONS_FILE = "publications_2020_2026.csv"   # one row per publication
AUTHORSHIP_FILE = "authorship_2020_2026.csv"       # one row per publication × researcher
SCENARIO = "broad_textual"            # all document types, both researcher categories, 2020–2026
YEARS = list(range(2020, 2027))
PARTIAL_YEAR = 2026             # latest year; extraction date is not documented locally
GROUPS = ["CIS", "PI2", "PI3", "SAD", "TD"]
MIN_EDGE_WEIGHT = 1             # main network: one shared publication is enough for an edge
SENSITIVITY_MIN_EDGE_WEIGHT = 2 # optional extension only (section 10)
LAYOUT_SEED = 42
SPRING_ITERATIONS = 100
PALETTE = {"CIS": "#0072B2", "PI2": "#D55E00", "PI3": "#E69F00", "SAD": "#009E73", "TD": "#CC79A7", "unresolved": "#999999"}
MARKERS = {"CIS": "o", "PI2": "s", "PI3": "P", "SAD": "^", "TD": "D", "unresolved": "X"}

**Reference values.** The course computed the same network once, with a different method
(sparse matrices, no NetworkX), and recorded the results with the fingerprint (SHA-256 hash) of
each input file. Your checkpoints compare your results with these values. A hash that differs
means that your data file is not the reference file: stop and tell the teacher; do not adjust
your results to match.

In [ ]:
REFERENCE = {
    "input_sha256": {
        PUBLICATIONS_FILE: "badca2d7219a3f5b14a2c9f6f7b100a4c094ffe0dcbe6431a966a0471c3b7362",
        AUTHORSHIP_FILE: "512f30215fe028935fc599667a6632e017f99f6d9f77f2892258e7f555c66338",
    },
    "n_publications": 3343,
    "n_authorship_rows_raw": 4363,
    "n_researchers": 187,
    "coauthorship_n_edges": 364,
    "coauthorship_total_edge_weight": 1337,
    "coauthorship_n_components": 33,
    "coauthorship_n_isolates": 29,
    "coauthorship_largest_component_n": 148,
    "intergroup_edge_weight": 430,
    "total_classifiable_edge_weight": 1337,
    "publication_multigroup_n": 266,
    "publication_group_denominator_n": 3343,
}

**What the literature says — the question and the universe.** A narrow, answerable question
decides which data you need and how you will read the result; a descriptive question asks what
the data show, not why (Lau, Gonzalez & Nolan (2023), *Learning Data Science*, “Ask a Question”, p. 4). Data that were assembled by someone else for other purposes
("found data") need an explicit argument for why they fit the question and a statement of what
they leave out (Grimmer, Roberts & Stewart (2022), *Text as Data*, “4.3 Considerations of "Found Data"”). When whole records are absent from what you can access, the gap can bias
the result, and more of the same data does not repair it (Lau, Gonzalez & Nolan (2023), *Learning Data Science*, “Missing Values and Records”, pp. 176–177). Here: Scopus only, GOVCOPP
researchers only, external coauthors absent, 2026 incomplete.

**Teacher note — universe.** The question is descriptive ("how is observed coauthorship
structured"), so the universe must be fixed before counting: scenario `broad_textual`, all document
types, both researcher categories, 2020–2026. The data are found data (a Scopus extraction for
teaching), so the group must say what they cannot see: publications outside Scopus, coauthors
outside GOVCOPP, the rest of 2026. The missing external coauthors are missing *records* in the
sense of Lau, Gonzalez & Nolan (2023), *Learning Data Science*, “Missing Values and Records”, pp. 176–177: they change degree, components and isolates, and adding more GOVCOPP rows would
not fix that. Typical error: treating the network as "GOVCOPP's collaboration" without the
boundary.

**Three distinctions to keep in mind all session.**

- *Fictitious example vs. real case.* Sections 1–3 use a made-up example (publications P1–P4,
  researchers A–E). Its numbers are **not** GOVCOPP results.
- *Raw rows vs. distinct relations.* A file can repeat a publication × researcher pair; the
  network counts each pair once.
- *Main network vs. optional extension.* The main network uses every shared publication; the
  extension in section 10 is another scenario and never replaces it.

**From the slides to this notebook.** Each idea from the first part of the class becomes a few
lines of code here. Use the slide title to find the explanation again.

| Slides | Idea | Section | Object in the code |
|---|---|---|---|
| «Three things, two tables»; «Where the data come from…»; «Audit first, then fix the universe» | units, provenance, audit before analysis | 4 | `publications`, `authorship`, `audit` |
| «Count before you project»; «The bipartite authorship network» | two kinds of node, links only between kinds | 1, 5 | `toy_bipartite`, `toy_incidence`, `B` |
| «Build the pairs»; «The coauthorship projection, step by step»; «"AB appears twice…"» | pairs, projection, edge weight | 2, 6 | `toy_pairs`, `edges`, `G` |
| «The same count as a matrix product»; «"Use B Bᵀ as it comes out"» | the matrix product and its diagonal | 2, 6 | `shared`, `M` |
| «Publications, degree, strength…»; «Components, isolates and density» | three measures per researcher; the shape of the network | 3, 7 | `toy_measures`, `nodes`, `network_summary` |
| «What the projection forgets» | what is lost when publications disappear | 2, 6 | `team_sizes` |
| «Ties across groups: two shares, two units»; «Two shares, and two ways of counting publications» | units and denominators; whole and fractional counts | 8 | `edge_groups`, `by_group` |
| «Optional: repeated ties only» | another scenario, not a correction | 10 | `G2` |

## 1 Fictitious example: the bipartite authorship network

*Fictitious data, written for this exercise. Not GOVCOPP data.*

| Publication | Researchers |
|---|---|
| P1 | A, B |
| P2 | A, B, C |
| P3 | C, D |
| P4 | E |

A, B and C belong to the fictitious group G1; D and E to G2.

**Before running anything:** on paper, draw publications on one line and researchers on
another, and join each researcher to each publication they sign. How many nodes? How many lines?

In [ ]:
toy = pd.DataFrame(
    [("P1", "A"), ("P1", "B"), ("P2", "A"), ("P2", "B"), ("P2", "C"), ("P3", "C"), ("P3", "D"), ("P4", "E")],
    columns=["publication_id", "researcher_id"],
)
TOY_GROUP = {"A": "G1", "B": "G1", "C": "G1", "D": "G2", "E": "G2"}   # fictitious groups
toy

**Exercise 1.** Count the nodes of the bipartite network (researchers plus publications) and
its links (distinct publication × researcher pairs). One line each.

In [ ]:
n_toy_nodes = toy["researcher_id"].nunique() + toy["publication_id"].nunique()
n_toy_links = len(toy.drop_duplicates(["publication_id", "researcher_id"]))
n_toy_nodes, n_toy_links

In [ ]:
# Checkpoint — your paper drawing and the code must agree.
assert n_toy_nodes == 9, "Count researchers and publications separately, then add them."
assert n_toy_links == 8, "One link per row of the authorship table, without repeated pairs."

**In symbols.** Write $R$ for the set of researchers and $P$ for the set of publications. The
bipartite network has the nodes $R \cup P$ (read: "R union P", both kinds together) and one link
for every researcher–publication pair in which the researcher signs the publication. The
**incidence matrix** $B$ stores the same links as numbers, one row per researcher and one column
per publication:

$$B_{rp} = \begin{cases} 1 & \text{if researcher } r \text{ signs publication } p \\ 0 & \text{otherwise} \end{cases}$$

Read: "B r p is one when r signs p". The number of nodes is $\lvert R \rvert + \lvert P \rvert$ (the
vertical bars mean "how many"), and the number of links is the number of ones in $B$. In GOVCOPP,
$R$ holds the pseudonymised researchers, $P$ the publications of the textual universe, and each
one in $B$ is one row of the authorship table.

The same relation as a graph with two kinds of node. The prefixes `R:` and `P:` stop a
researcher and a publication from ever sharing a name. The **incidence matrix** is the same
information as a table: researchers in the rows, publications in the columns, 1 if the
researcher signs the publication and 0 otherwise.

In [ ]:
toy_bipartite = nx.Graph()
toy_bipartite.add_nodes_from(("R:" + r for r in sorted(TOY_GROUP)), kind="researcher")
toy_bipartite.add_nodes_from(("P:" + p for p in sorted(toy["publication_id"].unique())), kind="publication")
toy_bipartite.add_edges_from(("R:" + r, "P:" + p) for p, r in toy.itertuples(index=False))

toy_incidence = pd.crosstab(toy["researcher_id"], toy["publication_id"])
print(toy_bipartite.number_of_nodes(), "nodes,", toy_bipartite.number_of_edges(), "authorship links")
toy_incidence

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.8))
pos = {f"P:{p}": (i * 1.3 + 0.65, 1) for i, p in enumerate(["P1", "P2", "P3", "P4"])}
pos.update({f"R:{r}": (i * 1.3, 0) for i, r in enumerate(["A", "B", "C", "D", "E"])})
nx.draw_networkx_edges(toy_bipartite, pos, ax=ax, edge_color="#B8B8B8", width=2)
nx.draw_networkx_nodes(toy_bipartite, pos, nodelist=[n for n in pos if n.startswith("P:")], node_shape="s",
                       node_color="#4D4D4D", node_size=700, ax=ax)
nx.draw_networkx_nodes(toy_bipartite, pos, nodelist=[n for n in pos if n.startswith("R:")], node_color="white",
                       edgecolors="#222222", node_size=700, ax=ax)
nx.draw_networkx_labels(toy_bipartite, pos, labels={n: n[2:] for n in pos}, ax=ax)
ax.set_title("Fictitious example: authorship (publications above, researchers below)")
ax.axis("off")
plt.show()

**Your answer.** What is the unit of one row of `toy`? And of one row of `toy_incidence`?

*Write here.*

**Teacher note.** A row of `toy` is one authorship link: one researcher signing one publication
(publication × researcher). A row of `toy_incidence` is one researcher, with one column per
publication. Typical error: calling a row of the authorship table "a publication". P2 has three
rows, but it is one publication.

## 2 From authorship to coauthorship: the projection

Two researchers are **coauthors** if they sign at least one publication together. The
**projection** keeps only researchers and joins two of them when they share a publication. The
**weight** of the edge is the number of distinct publications they share.

We first build the pairs by hand, publication by publication.

In [ ]:
toy_pairs = Counter()
for publication, researchers in toy.groupby("publication_id")["researcher_id"]:
    for a, b in itertools.combinations(sorted(researchers), 2):   # every unordered pair, once
        toy_pairs[(a, b)] += 1
dict(toy_pairs)

**Exercise 2.** List the unordered pairs generated by P2 alone. Use `itertools.combinations`
on the sorted researchers of P2.

In [ ]:
p2_researchers = sorted(toy.loc[toy["publication_id"] == "P2", "researcher_id"])
p2_pairs = list(itertools.combinations(p2_researchers, 2))
p2_pairs

In [ ]:
# Checkpoint — a publication with three researchers gives 3 × 2 / 2 = 3 pairs.
assert len(p2_pairs) == len(p2_researchers) * (len(p2_researchers) - 1) // 2, "Each unordered pair once: AB and BA are the same pair."
assert set(p2_pairs) == {("A", "B"), ("A", "C"), ("B", "C")}

AB appears in P1 and again in P2. The repetition **raises the weight** of the AB edge to 2; it
does not create a second AB edge. And E, alone in P4, generates no pair at all, but E is still a
researcher in the network.

The same result with a matrix product. Multiplying the incidence matrix by its transpose,
`B @ B.T`, counts for every pair of researchers how many publications both sign. On the
diagonal, a researcher is paired with themself: that cell counts their publications. We keep
the diagonal as the number of publications and then set it to zero, because a researcher is not
their own coauthor.

**In symbols.** $B^\top$ ("B transposed") is $B$ with rows and columns swapped. The product

$$C = B\,B^\top, \qquad C_{ij} = \sum_{p \in P} B_{ip}\,B_{jp}$$

reads: "for researchers $i$ and $j$, go through every publication $p$ and add one when both sign
it". The product $B_{ip}\,B_{jp}$ is 1 only when both entries are 1, so the sum counts shared
publications. Off the diagonal ($i \neq j$), $C_{ij}$ is the edge weight $w_{ij}$. On the
diagonal, $C_{ii} = \sum_p B_{ip}$ is the number of publications of $i$: we keep it as its own
column and then set it to zero.

Seen from one publication: a publication signed by $n_p$ researchers adds one unit of weight to
each of its $\binom{n_p}{2} = n_p(n_p - 1)/2$ pairs (read: "$n_p$ choose 2", the number of
different pairs among $n_p$ people). P2 has $n_p = 3$ and adds 3 pairs; P4 has $n_p = 1$ and adds
none. You will use this rule again in section 6.

In [ ]:
B_toy = toy_incidence.to_numpy(dtype="int64")          # int64, not 8-bit integers: counts can grow
shared = B_toy @ B_toy.T
toy_publications = pd.Series(np.diag(shared), index=toy_incidence.index, name="n_publications")
np.fill_diagonal(shared, 0)
pd.DataFrame(shared, index=toy_incidence.index, columns=toy_incidence.index)

In [ ]:
# Checkpoint — two independent methods must give the same edges and weights.
from_matrix = {(a, b): int(shared[i, j]) for i, a in enumerate(toy_incidence.index)
               for j, b in enumerate(toy_incidence.index) if i < j and shared[i, j] > 0}
assert from_matrix == dict(toy_pairs), "The matrix product and the pair enumeration disagree: check the diagonal."
assert toy_publications.to_dict() == {"A": 2, "B": 2, "C": 2, "D": 1, "E": 1}

**Your answer.** P2 makes A, B and C a triangle in the projection. Does the triangle tell you
that there were three separate collaborations? What information about the original publications
can you no longer read from the projection?

*Write here.*

**Teacher note.** No. One three-author publication produces the whole triangle: the projection
turns one event into three pairs. From the projection alone you can no longer tell which edges
came from the same publication, how many researchers each publication had, or that P4 exists
(E has no edge). That information stays in the authorship table, which is why we keep it.

## 3 Degree, strength, components and isolates

- **Degree**: the number of *distinct* coauthors of a researcher.
- **Strength**: the sum of the weights of their edges (each shared publication counted once
  per coauthor).
- **Publications**: how many publications they sign. A different quantity again.
- A **component** is a set of researchers linked by some path of edges; an **isolate** is a
  researcher with no edge.
- **Density** = distinct edges / possible pairs of researchers, *counting the isolates* among
  the researchers.

**In symbols.** With $w_{ij}$ the weight of the edge between $i$ and $j$ (zero when there is no
edge) and $N$ the number of researchers:

$$k_i = \#\{\, j \neq i : w_{ij} > 0 \,\} \qquad s_i = \sum_{j \neq i} w_{ij} \qquad D = \frac{\lvert E \rvert}{\binom{N}{2}} = \frac{2\,\lvert E \rvert}{N(N-1)}$$

Read: the **degree** $k_i$ is "how many $j$ have a positive weight with $i$"; the **strength**
$s_i$ is "the sum of $i$'s weights"; the **density** $D$ is "distinct edges $\lvert E \rvert$ over
the number of possible pairs". Every edge weight is counted twice in the strengths, once at each
end, so $\sum_i s_i = 2 \sum_{i<j} w_{ij}$. That is a check you can run on any weighted network.

None of the three measures says anything about quality, and degree 0 only means "no coauthor
*among the researchers we observe*".

In [ ]:
G_toy = nx.Graph()
G_toy.add_nodes_from(sorted(TOY_GROUP))                    # all researchers, E included
G_toy.add_weighted_edges_from((a, b, w) for (a, b), w in sorted(toy_pairs.items()))
toy_measures = pd.DataFrame({
    "n_publications": toy_publications,
    "degree": dict(G_toy.degree()),
    "strength": dict(G_toy.degree(weight="weight")),
}).sort_index()                                            # by ID, never by "importance"
toy_measures

**Exercise 3.** Compute the density of the fictitious network from its numbers of edges and
researchers (do not call `nx.density`).

In [ ]:
n_toy = G_toy.number_of_nodes()
toy_density = G_toy.number_of_edges() / (n_toy * (n_toy - 1) / 2)
toy_density

In [ ]:
# Checkpoint — 4 edges out of 10 possible pairs.
assert math.isclose(toy_density, nx.density(G_toy)), "Possible pairs among n researchers: n × (n − 1) / 2, isolates included."
assert nx.number_connected_components(G_toy) == 2 and nx.number_of_isolates(G_toy) == 1

**Your answers.**

1. A has 2 publications, degree 2 and strength 3. Explain each number in one sentence.
2. E has degree 0. What can you conclude about E, and what can you **not** conclude?

*Write here.*

**Teacher note.** (1) A signs P1 and P2 (2 publications); A has two distinct coauthors, B and C
(degree 2); A shares two publications with B and one with C (strength 2 + 1 = 3). (2) E signs a
publication in these data but shares none with another researcher *in this network*. We cannot
conclude that E does not collaborate: P4 may have coauthors outside the observed group, and the
data only contain the researchers we selected. Typical error: reading strength as "number of
publications" or as quality.

## 4 Load and audit the real data — checkpoint C1 (minutes 95–120)

**Provenance.** Scientific production of GOVCOPP researchers indexed in Scopus, 2020–2026. The two local CSV files were provided with this project; the extraction date and redistribution rights are not documented here. The 2026 records may be incomplete. Researchers are pseudonymised (`R001`, …), but DOIs and titles can identify authors: do not publish tables with individual researchers without checking the data-use terms.

- `publications_2020_2026.csv`: one row per **publication**.
- `authorship_2020_2026.csv`: one row per **publication × researcher** link.

Every column is read as text, so that IDs keep their leading zeros; only an empty cell counts as missing (an empty cell is not a zero).


In [ ]:
from icd_project.data import read_table

publications = read_table(DATA_DIR / PUBLICATIONS_FILE)
authorship = read_table(DATA_DIR / AUTHORSHIP_FILE)
publications["year"] = pd.to_numeric(publications["year"]).astype("Int64")
print("publications:", publications.shape, " authorship:", authorship.shape)
authorship.head()


In [ ]:
# Are these the reference files? Same bytes, same fingerprint.
input_sha256 = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
                for name in (PUBLICATIONS_FILE, AUTHORSHIP_FILE)}
assert input_sha256 == REFERENCE["input_sha256"], "Input bytes differ from this project reference; verify the source files before continuing."

**What the literature says — the audit.** Before analysing a table, establish what one row
stands for (its granularity) and confirm it, for example by comparing the number of rows with
the number of distinct identifiers (Lau, Gonzalez & Nolan (2023), *Learning Data Science*, “Table Shape and Granularity”, pp. 153–156). Then check counts you can work out in advance and
cross-check anything odd against other columns or tables (Lau, Gonzalez & Nolan (2023), *Learning Data Science*, “Quality Checks”, p. 167).

**Audit note.** The publications table should contain 3343 distinct publication IDs; the authorship table should contain 4363 distinct publication × researcher pairs. Every link must point to an existing publication. Section 8 checks the groups against `govcopp_groups`. The local files include one PI3 researcher, while the preserved teacher solution uses a later PI3 → PI2 correction; do not transfer its group-level values to this version.


**The audit.** Before building anything, we check the keys. A missing ID, a repeated
publication, or a link to a publication that does not exist would make the network wrong, so
these checks block the analysis. A repeated publication × researcher pair would be reported and
counted once.

In [ ]:
key = ["publication_id", "researcher_id"]
audit = pd.Series({
    "publications (rows)": len(publications),
    "publication_id missing": publications["publication_id"].isna().sum(),
    "publication_id repeated": publications["publication_id"].duplicated().sum(),
    "authorship rows (raw)": len(authorship),
    "authorship key missing": authorship[key].isna().any(axis=1).sum(),
    "publication × researcher pair repeated": authorship.duplicated(key).sum(),
    "links to a publication that does not exist": (~authorship["publication_id"].isin(publications["publication_id"])).sum(),
    "publications without any link": (~publications["publication_id"].isin(authorship["publication_id"])).sum(),
    "years outside 2020–2026": (~publications["year"].isin(YEARS)).sum(),
    f"publications in {PARTIAL_YEAR} (partial year)": (publications["year"] == PARTIAL_YEAR).sum(),
}, name="value")
audit

In [ ]:
# Blocking problems: if one of these is not zero, stop and report it.
blocking = ["publication_id missing", "publication_id repeated", "authorship key missing",
            "links to a publication that does not exist", "years outside 2020–2026"]
assert audit[blocking].sum() == 0, f"Blocking audit problems: {audit[blocking][audit[blocking] > 0].to_dict()}"

**Formal groups.** A researcher has a formal group only if all their rows give one single, known
group. Two different groups (or none) leave the researcher *unresolved*: they stay in the
network, drawn in grey, and their edges are left out of the group comparison. We never pick the
first or the most frequent group.

In [ ]:
groups_per_researcher = authorship.groupby("researcher_id")["formal_group"].agg(lambda s: sorted(set(s.dropna())))
affiliation = pd.DataFrame({
    "formal_group": groups_per_researcher.map(lambda g: g[0] if len(g) == 1 and g[0] in GROUPS else None),
    "affiliation_status": groups_per_researcher.map(
        lambda g: "valid" if len(g) == 1 and g[0] in GROUPS else ("missing" if not g else "unresolved")),
})
print(affiliation["affiliation_status"].value_counts().to_dict())
print(affiliation["formal_group"].value_counts().to_dict())
authorship["category"].value_counts()

**Exercise 4.** Count the distinct publication × researcher links and the distinct researchers.

In [ ]:
n_links_unique = len(authorship.drop_duplicates(key))
n_researchers = authorship["researcher_id"].nunique()
n_links_unique, n_researchers

In [ ]:
# Checkpoint — compare with the case sheet.
assert len(publications) == REFERENCE["n_publications"], "Publications: count rows of the publications table, not of authorship."
assert len(authorship) == REFERENCE["n_authorship_rows_raw"]
assert n_links_unique <= len(authorship), "Distinct links cannot exceed raw rows."
assert n_researchers == REFERENCE["n_researchers"], "Count distinct researcher_id values, not rows."

**Joins change the row count, not the number of publications.** Adding the publication year to
the authorship table gives one row per *link*. A publication with three GOVCOPP researchers
appears three times. Counting rows after this join answers a different question.

In [ ]:
links_with_year = authorship.merge(publications[["publication_id", "year"]], on="publication_id",
                                   how="left", validate="many_to_one")
print(len(links_with_year), "rows after the join;", links_with_year["publication_id"].nunique(), "distinct publications")

**A checkpoint inside the join.** `validate="many_to_one"` states what the join must look like:
many authorship rows for one publication. If a `publication_id` appeared twice in
`publications`, a plain join would silently multiply the rows. With `validate`, pandas stops
with an error instead. It is the W05 idea of a workflow checkpoint, written into the line that
could go wrong. In the same way, the fingerprint check above guards the W05 rule that raw data
are never edited: a CSV saved again through Excel gets a different fingerprint.

**Checkpoint C1 — record it before you go on.**

| Quantity | Your value |
|---|---|
| publications | |
| authorship rows (raw) | |
| distinct publication × researcher links | |
| researchers | |
| problems found (or "none") | |

**Checkpoint C1 reference for the local files.** 3343 publications, 4363 raw rows, 4363 distinct links, 187 researchers; no blocking problem. All 187 researchers have one valid formal group: CIS 73, PI2 35, PI3 1, SAD 30, TD 48. There are 386 publications dated 2026; completeness is unknown without an extraction date.


## 5 The real bipartite authorship network — checkpoint C2

Same construction as the fictitious example, now with 187 researchers and 3343 publications.
The incidence matrix has 187 × 3343 cells, almost all zeros, so we store it as a **sparse**
matrix: only the ones are kept. We do not draw the bipartite network: 3530 nodes are unreadable.

In [ ]:
links = authorship.drop_duplicates(key)                          # binary relation: each pair once
researchers = sorted(links["researcher_id"].unique())            # every researcher, isolates included
publication_ids = sorted(publications["publication_id"])
row = {r: i for i, r in enumerate(researchers)}
col = {p: j for j, p in enumerate(publication_ids)}
B = sp.csr_matrix(
    (np.ones(len(links), dtype="int64"), (links["researcher_id"].map(row), links["publication_id"].map(col))),
    shape=(len(researchers), len(publication_ids)), dtype="int64",
)
print("incidence matrix:", B.shape, "| ones:", B.nnz, f"| share of cells that are 1: {B.nnz / (B.shape[0] * B.shape[1]):.4f}")

**A small real extract.** One publication with exactly three GOVCOPP researchers, as in P2. It
is an illustration chosen by its ID, not a summary of the network.

In [ ]:
per_publication = links.groupby("publication_id").size()
example_id = per_publication[per_publication == 3].index[0]
links.loc[links["publication_id"] == example_id, ["publication_id", "researcher_id", "formal_group"]]

**Exercise 5.** Count the nodes of the real bipartite network from the shape of `B`.

In [ ]:
bipartite_n_nodes = B.shape[0] + B.shape[1]
bipartite_n_nodes

In [ ]:
# Checkpoint C2 — every link is in the matrix once, and no link lost one of its ends.
assert bipartite_n_nodes == n_researchers + len(publications), "Nodes are researchers plus publications."
assert B.sum() == n_links_unique and B.max() == 1, "Each publication × researcher pair must be a single 1."
assert (np.asarray(B.sum(axis=1)).ravel() >= 1).all(), "Every researcher signs at least one publication."
assert (np.asarray(B.sum(axis=0)).ravel() >= 1).all(), "Every publication has at least one GOVCOPP researcher."

## 6 The real coauthorship projection — checkpoint C3 (minutes 120–145)

Exactly the steps of section 2: `B @ B.T`, keep the diagonal as publications per researcher,
remove it, and build a simple, undirected, weighted network with **all** researchers.

In [ ]:
M = (B @ B.T).tocsr()
n_publications_per_researcher = M.diagonal().astype("int64")
M.setdiag(0)
M.eliminate_zeros()                                              # drop the zeros left on the diagonal
upper = sp.triu(M, k=1).tocoo()                                  # each unordered pair once
edges = pd.DataFrame({
    "researcher_a": [researchers[i] for i in upper.row],
    "researcher_b": [researchers[j] for j in upper.col],
    "weight": upper.data.astype("int64"),
}).sort_values(["researcher_a", "researcher_b"], ignore_index=True)

G = nx.Graph()
G.add_nodes_from(researchers)                                    # nodes first: isolates stay
G.add_weighted_edges_from(edges.itertuples(index=False))
print(G.number_of_nodes(), "researchers,", G.number_of_edges(), "edges, total weight", edges["weight"].sum())

**Exercise 6.** Each publication with *n* GOVCOPP researchers contributes *n*(*n* − 1)/2 pairs.
Sum this over all publications, using `per_publication` (researchers per publication).
External coauthors are not in the data, so *n* counts only GOVCOPP researchers.

In [ ]:
pairs_from_publications = int((per_publication * (per_publication - 1) // 2).sum())
pairs_from_publications

In [ ]:
# Checkpoint C3 — the network keeps its units.
total_weight = int(edges["weight"].sum())
assert pairs_from_publications == total_weight, "Total edge weight must equal the pairs generated publication by publication."
assert M.sum() == 2 * total_weight, "The matrix counts each pair twice (AB and BA)."
assert nx.number_of_selfloops(G) == 0, "A researcher is not their own coauthor: remove the diagonal."
assert G.number_of_nodes() == n_researchers, "Every researcher stays in the network, isolates included."
assert (edges["weight"] >= 1).all() and edges["weight"].dtype == "int64"
assert G.number_of_edges() == REFERENCE["coauthorship_n_edges"] and total_weight == REFERENCE["coauthorship_total_edge_weight"]

**What checkpoint C3 has just shown.** The rule of section 2, now on the real data:

$$\sum_{i<j} w_{ij} \;=\; \sum_{p \in P} \binom{n_p}{2} \qquad\text{and}\qquad \sum_{i \neq j} C_{ij} \;=\; 2 \sum_{i<j} w_{ij}$$

The left side of the first identity is a property of the **network** (the total edge weight);
the right side is a property of the **publications** (how many GOVCOPP researchers each one
has). When they agree, no pair was lost or counted twice between the authorship table and the
network. This is what the slide «Demo 3: the projection keeps its units» means. The second
identity is why the code keeps only the upper triangle (`sp.triu`): the full matrix holds each
pair twice, AB and BA. If C3 fails, look first for a repeated link or a diagonal left in.

**How many publications add nothing to the projection?**

In [ ]:
team_sizes = per_publication.value_counts().sort_index().rename_axis("GOVCOPP researchers per publication").rename("publications")
team_sizes.to_frame().assign(pairs_generated=lambda t: t["publications"] * t.index * (t.index - 1) // 2)

**Teacher note.** 2555 of the 3343 publications have a single observed GOVCOPP researcher and
generate no pair: most of the production is invisible in the projection. This is a property of
the network boundary (only GOVCOPP researchers are observed), not evidence that those authors
work alone.

## 7 Measures and figures of the observed network

All measures are computed on the **complete** main network: 187 researchers, every shared
publication. The table is ordered by ID. It is not a ranking, and we do not build one.

In [ ]:
nodes = pd.DataFrame({
    "researcher_id": researchers,
    "n_publications": n_publications_per_researcher,
    "degree": [G.degree(r) for r in researchers],
    "strength": [G.degree(r, weight="weight") for r in researchers],
}).merge(affiliation, left_on="researcher_id", right_index=True)
components = sorted(nx.connected_components(G), key=lambda c: (-len(c), min(c)))
component_of = {r: k for k, c in enumerate(components, start=1) for r in c}
nodes["component"] = nodes["researcher_id"].map(component_of)
network_summary = pd.Series({
    "researchers (nodes)": G.number_of_nodes(),
    "distinct edges": G.number_of_edges(),
    "total edge weight": total_weight,
    "density": round(nx.density(G), 4),
    "components": len(components),
    "largest component": len(components[0]),
}, dtype=object)
network_summary

**Exercise 7.** Count the isolates: researchers with degree 0.

In [ ]:
n_isolates = int((nodes["degree"] == 0).sum())
n_isolates

In [ ]:
# Checkpoint — isolates are part of the network, and of its denominators.
assert n_isolates == nx.number_of_isolates(G) == REFERENCE["coauthorship_n_isolates"], "An isolate has degree 0 and stays in the node table."
assert len(components) == REFERENCE["coauthorship_n_components"] and len(components[0]) == REFERENCE["coauthorship_largest_component_n"]
assert math.isclose(nx.density(G), G.number_of_edges() / (n_researchers * (n_researchers - 1) / 2))

**Three different quantities.** Publications, degree and strength describe different things:
volume of publishing, number of distinct partners, and frequency of joint publishing.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, column, label in zip(axes, ["n_publications", "degree", "strength"],
                             ["publications per researcher", "degree: distinct GOVCOPP coauthors",
                              "strength: sum of edge weights"]):
    ax.hist(nodes[column], bins=min(25, nodes[column].max() + 1), color="#4D7EA8", edgecolor="white")
    ax.axvline(nodes[column].median(), color="#222222", linestyle="--", linewidth=1)
    ax.set_xlabel(label)
axes[0].set_ylabel("number of researchers")
fig.suptitle(f"The same {n_researchers} researchers, three quantities (scenario {SCENARIO}, 2020–2026)")
fig.tight_layout()
(OUTPUT_DIR / "coauthorship").mkdir(exist_ok=True)
fig.savefig(OUTPUT_DIR / "coauthorship" / "degree_strength_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

**The network.** One spring layout for the largest component (seed 42, 100 iterations). The
researchers outside it are counted below the figure, so the figure never hides them. Colour and
shape show the formal group; node size is the same for everyone. **Position is not a measure**:
a different seed moves the nodes without changing any degree.

In [ ]:
largest = G.subgraph(components[0])
pos = nx.spring_layout(largest, seed=LAYOUT_SEED, iterations=SPRING_ITERATIONS, weight="weight")
group = nodes.set_index("researcher_id")["formal_group"].fillna("unresolved")
fig, ax = plt.subplots(figsize=(9, 8))
nx.draw_networkx_edges(largest, pos, ax=ax, edge_color="#B8B8B8",
                       width=[0.4 + 0.6 * np.log2(d["weight"]) for _, _, d in largest.edges(data=True)])
for g in GROUPS + ["unresolved"]:
    members = [r for r in largest if group[r] == g]
    if members:
        nx.draw_networkx_nodes(largest, pos, nodelist=members, node_color=PALETTE[g], node_shape=MARKERS[g],
                               node_size=45, edgecolors="white", linewidths=0.6, label=g, ax=ax)
outside = n_researchers - len(components[0])
ax.set_title(f"Largest component: {len(components[0])} of {n_researchers} researchers "
             f"({outside} outside it: {len(components) - 1 - n_isolates} small components and {n_isolates} isolates)")
ax.legend(title="formal group", loc="upper left", frameon=False)
ax.axis("off")
fig.savefig(OUTPUT_DIR / "coauthorship" / "largest_component.png", dpi=150, bbox_inches="tight")
plt.show()

**Your answers.**

1. Describe the network in two sentences using numbers from `network_summary` (with units).
2. A classmate says: "The isolated researchers do not collaborate, and nodes of the same colour
   that sit close together share research themes." Correct both claims.

*Write here.*

**Teacher note.** (1) Example: 187 researchers, 364 distinct coauthor pairs with a total weight
of 1337 shared-publication contributions; density 0.0209, i.e. about 2 % of the possible pairs
share at least one publication. 148 researchers (79 %) are in one connected component; 29 are
isolates and 10 sit in 3 small components. (2) An isolate has no GOVCOPP coauthor *in these
data*: they may coauthor with people outside GOVCOPP or outside the indexed sources. Position
comes from a layout algorithm: closeness on the page is not a thematic measure, and colour is the
recorded formal group, not a finding. Themes are measured from abstracts in W07–W09.

## 8 Coauthorship and the formal groups — checkpoint C4 (minutes 145–170)

**What the literature says — the unit.** The unit of analysis is a choice: it should follow
from the question being asked, from the method, and from the form in which the data arrive
(Grimmer, Roberts & Stewart (2022), *Text as Data*, “5.2 Choose the Unit of Analysis”). Below, the same source records give two shares because they are read in two
representations, at two units: the projection (researcher pairs) and the authorship links
(publications).

Two different questions, with **different units and denominators**:

- *Edges* (from the projection): of all the edge weight between researchers whose group is valid,
  how much joins researchers of **different** groups? Unit: researcher pair × shared publication.
- *Publications* (from the authorship links): of all publications whose researchers all have a
  valid group, how many involve **two or more** groups? Unit: publication. The projection alone
  cannot answer this: it no longer knows which edges came from the same publication.

In the fictitious example the first share is 1/5 (only CD crosses G1–G2) and the second is 1/4
(only P3 involves both groups). P4 belongs to G2 alone but contains no G2 pair: a publication
exclusive to a group is not coauthorship inside the group.

**In symbols.** Write $g_i$ for the formal group of researcher $i$, $G_p$ for the set of formal
groups present in publication $p$, and $\mathbf{1}[\ldots]$ for "1 when the condition in brackets
holds, 0 otherwise". The two shares are:

$$\text{edge share} = \frac{\sum_{i<j} w_{ij}\,\mathbf{1}[g_i \neq g_j]}{\sum_{i<j} w_{ij}} \qquad\qquad \text{publication share} = \frac{\#\{\, p : \lvert G_p \rvert \geq 2 \,\}}{\lvert P \rvert}$$

Read the first as "weight between different groups over all weight", with sums only over pairs
whose two researchers have a valid group. Read the second as "publications with two or more
groups over all publications" (the denominator only counts publications whose researchers all
have a valid group). The numerators differ, the denominators differ, and so do the units:
researcher pair × shared publication against publication. That is why the two numbers are not
two estimates of the same thing, and why your sentence must name the unit.

In [ ]:
edge_groups = edges.assign(group_a=edges["researcher_a"].map(affiliation["formal_group"]),
                           group_b=edges["researcher_b"].map(affiliation["formal_group"]))
edge_groups["classifiable"] = edge_groups["group_a"].notna() & edge_groups["group_b"].notna()
edge_groups["between_groups"] = edge_groups["classifiable"] & (edge_groups["group_a"] != edge_groups["group_b"])
unclassifiable_weight = int(edge_groups.loc[~edge_groups["classifiable"], "weight"].sum())

pair = edge_groups[edge_groups["classifiable"]].assign(
    pair=lambda t: [" – ".join(sorted(x)) for x in zip(t["group_a"], t["group_b"])])
group_pairs = pair.groupby("pair").agg(researcher_pairs=("weight", "size"), sum_of_weights=("weight", "sum"))
print("Unit: pairs of researchers with at least one shared publication; weight = shared publications.")
print("Edge weight left out (unresolved affiliation):", unclassifiable_weight)
group_pairs

**Exercise 8.** Compute the between-group share of edge weight: the sum of weights of the
between-group edges divided by the sum of weights of the classifiable edges.

In [ ]:
intergroup_edge_weight = int(edge_groups.loc[edge_groups["between_groups"], "weight"].sum())
total_classifiable_edge_weight = int(edge_groups.loc[edge_groups["classifiable"], "weight"].sum())
intergroup_edge_weight_share = intergroup_edge_weight / total_classifiable_edge_weight
intergroup_edge_weight, total_classifiable_edge_weight, round(intergroup_edge_weight_share, 4)

In [ ]:
# Checkpoint — a share has a numerator, a denominator and a unit.
assert 0 <= intergroup_edge_weight_share <= 1
assert total_classifiable_edge_weight + unclassifiable_weight == total_weight, "Classifiable and excluded weight must add up to the total."
assert intergroup_edge_weight == REFERENCE["intergroup_edge_weight"], "Between groups: both researchers have a valid group and the groups differ."

Now the publication view. A publication enters the denominator only if every one of its
GOVCOPP researchers has a valid group.

In [ ]:
link_groups = links.assign(group=links["researcher_id"].map(affiliation["formal_group"]))
complete = link_groups.groupby("publication_id")["group"].apply(lambda s: s.notna().all())
n_groups = link_groups.groupby("publication_id")["group"].nunique()
publication_group_denominator_n = int(complete.sum())
publication_multigroup_n = int(((n_groups >= 2) & complete).sum())
publication_multigroup_share = publication_multigroup_n / publication_group_denominator_n
print(f"{publication_multigroup_n} of {publication_group_denominator_n} publications involve ≥ 2 groups "
      f"({publication_multigroup_share:.4f}); {len(complete) - publication_group_denominator_n} left out")

In [ ]:
# Checkpoint — the supplied field govcopp_groups must tell the same story.
from_field = publications.set_index("publication_id")["govcopp_groups"].str.split("; ").str.len()
assert int((from_field >= 2).sum()) == publication_multigroup_n == REFERENCE["publication_multigroup_n"]

**Counting publications per group.** If each multigroup publication counts once *for every*
group it involves, the group totals add up to more than 3343. The **fractional publication
count** gives each publication a total of one, split equally among its groups, so the group
totals add back to the number of publications. (W08 uses the same rule for group topic profiles.)

**In symbols.** For a group $g$:

$$\text{whole: } n_g = \sum_{p \in P} \mathbf{1}[g \in G_p] \qquad\qquad \text{fractional: } f_g = \sum_{p \in P} \frac{\mathbf{1}[g \in G_p]}{\lvert G_p \rvert}$$

A publication with two groups adds 1 to each group's whole count, but only $\tfrac{1}{2}$ to each
fractional count. Summing over the groups, every publication contributes exactly one to the
fractional total, so $\sum_g f_g = \lvert P \rvert$, while $\sum_g n_g = \sum_p \lvert G_p \rvert$
exceeds $\lvert P \rvert$ whenever some publication has more than one group. The fractional count
is a way of *attributing* production to groups; it is not a sampling weight.

In [ ]:
group_incidence = link_groups.drop_duplicates(["publication_id", "group"]).dropna(subset=["group"])
group_incidence = group_incidence.assign(share=1 / group_incidence["publication_id"].map(n_groups))
by_group = group_incidence.groupby("group").agg(publications_whole=("publication_id", "size"),
                                                publications_fractional=("share", "sum"))
by_group

**Exercise 9.** Add up the whole counts and the fractional counts over the five groups.

In [ ]:
sum_whole = int(by_group["publications_whole"].sum())
sum_fractional = float(by_group["publications_fractional"].sum())
sum_whole, round(sum_fractional, 6)

In [ ]:
# Checkpoint — fractional attribution adds back to the number of publications.
assert sum_whole > len(publications), "Whole counting counts a multigroup publication once per group."
assert math.isclose(sum_fractional, len(publications), rel_tol=1e-9), "Each publication must add up to exactly one across its groups."

**Checkpoint C4 — your conclusion.** Write one sentence about collaboration between groups that
states the value, its unit, its denominator and one limitation.

*Write here.*

**Checkpoint C4 reference for the local files.** Between-group edge weight is 430 / 1337 = 0.3216 (unit: researcher pair × shared publication; all 1337 classifiable, 0 excluded). Multigroup publications are 266 / 3343 = 0.0796 (unit: publication; 0 excluded). Whole group counts sum to 3616; fractional counts sum to 3343.0. These differ from the teacher solution because the local files retain one PI3 affiliation. Neither share shows causality, interdisciplinarity or successful integration. The supplied group matrix has exclusive publications on its diagonal: that is not within-group coauthorship.


**Where today's numbers go in the project.**

- **Common trunk (phases 1–2).** The audit of section 4 and the shares of section 8 are your
  evidence for the first two phases of the project, and they enter the M2 dossier. Record each
  conclusion as entry **E06** of `docs/evidence_register.md`: the claim, the output that supports
  it, the unit and denominator, the check you ran, one limitation, and any AI use.
- **The project question.** Coauthorship describes *who signs with whom*. It does not measure
  themes, so it cannot answer the project question on its own. W07–W09 build the thematic side.
- **Variants.** This notebook does not do any variant's work. V1 needs the `journal_core`
  universe, which this notebook does not read. V4 needs publications shared *between pairs of
  groups* and a thematic proximity that only exists from W09. The `group_pairs` table above
  counts researcher pairs, which is a different unit. If your group is considering one of these
  variants, write in the Research Plan which unit and denominator it will need. Do not compute
  it yet.

## 9 Group synthesis and outputs (minutes 145–180)

**Synthesis (180–250 words).** Use these five headings.

1. *Question and network.* The descriptive question and the definition of the network you
   analysed (nodes, edges, weight, universe).
2. *Two conclusions*, each with a number and its unit or denominator.
3. *Two limitations* that affect those conclusions.
4. *A future question* about thematic structure that coauthorship cannot answer.
5. *Changes and AI use.* What you changed in the code and why; any AI assistance, recorded below.

*Write here.*

**Decision log.** In the *Reason* column, connect each choice to the criterion quoted in this
notebook and to your own result.

| Decision | Alternative considered | Reason | Effect on the result |
|---|---|---|---|
| Scenario `broad_textual`, all document types | `journal_core` (variant V1) | | |
| Edge weight = distinct shared publications | fractional counting per pair | | |
| Keep isolates | drop researchers without edges | | |
| | | | |

**AI use log** (course policy).

| Tool and date | Request | Change made | How it was checked |
|---|---|---|---|
| | | | |

**Teacher note — reading the synthesis.** A good synthesis (i) defines the network (187
researchers, edges = at least one shared publication, weight = distinct shared publications,
broad_textual scenario 2020–2026 with 2026 completeness unknown); (ii) gives two conclusions whose numbers carry a
unit and a denominator, and does not mix the two shares; (iii) names limitations that touch
those conclusions: no external coauthors, Scopus coverage, current affiliation without
membership history, unequal group sizes, full counting inflating large teams; (iv) asks a
thematic question for W07–W09; (v) records changes and AI use. Weak syntheses rank individuals,
read layout distance as similarity, or call the shares interdisciplinarity.

**What the literature says — the evidence record.** Record enough about the procedure that
another person, or yourself months later, can understand and repeat it (The Turing Way Community (2022), *The Turing Way*, “Experimental Work”). For data you did
not produce, the record should say where they come from and exactly which version was analysed
(Irving et al. (2022), *Research Software Engineering with Python*, “13.1 Data Provenance”): here, the file names, their SHA-256 hashes, and that the extraction date is not documented.

**Teacher note — evidence record.** `statistics.json` carries the hashes of both inputs, the
parameters and the library versions; with the notebook and the commit, that is the record. A
group that copied a CSV through Excel will have a different hash: the record makes the change
visible instead of letting the numbers drift silently.

**Save the outputs.** The tables are ordered by ID and contain no ranking. `statistics.json`
records the inputs (with their hashes), the parameters, the library versions and the results.

In [ ]:
edges.to_csv(OUTPUT_DIR / "coauthorship" / "coauthorship_edges.csv", index=False)
nodes.sort_values("researcher_id").to_csv(OUTPUT_DIR / "coauthorship" / "researcher_nodes.csv", index=False)
group_pairs.to_csv(OUTPUT_DIR / "coauthorship" / "group_researcher_pairs.csv")
audit.to_csv(OUTPUT_DIR / "coauthorship" / "audit.csv")
statistics = {
    "n_publications": len(publications), "n_authorship_rows_raw": len(authorship),
    "n_authorship_links_unique": n_links_unique, "n_researchers": n_researchers,
    "bipartite_n_nodes": bipartite_n_nodes, "bipartite_n_edges": int(B.sum()),
    "coauthorship_n_nodes": G.number_of_nodes(), "coauthorship_n_edges": G.number_of_edges(),
    "coauthorship_total_edge_weight": total_weight, "coauthorship_density": nx.density(G),
    "coauthorship_n_components": len(components), "coauthorship_n_isolates": n_isolates,
    "coauthorship_largest_component_n": len(components[0]),
    "intergroup_edge_weight": intergroup_edge_weight, "total_classifiable_edge_weight": total_classifiable_edge_weight,
    "unclassifiable_edge_weight": unclassifiable_weight, "intergroup_edge_weight_share": intergroup_edge_weight_share,
    "publication_multigroup_n": publication_multigroup_n,
    "publication_group_denominator_n": publication_group_denominator_n,
    "publication_multigroup_share": publication_multigroup_share,
    "input_file_hashes": input_sha256, "dependency_versions": VERSIONS,
    "parameters": {"scenario": SCENARIO, "years": YEARS, "partial_year": PARTIAL_YEAR,
                   "edge_weight": "distinct_shared_publications", "counting": "full_per_researcher_pair",
                   "min_edge_weight": MIN_EDGE_WEIGHT, "keep_isolates": True, "remove_self_loops": True,
                   "layout_seed": LAYOUT_SEED, "spring_layout_iterations": SPRING_ITERATIONS},
}
(OUTPUT_DIR / "coauthorship" / "statistics.json").write_text(json.dumps(statistics, indent=2), encoding="utf-8")
sorted(p.name for p in (OUTPUT_DIR / "coauthorship").iterdir())

**What the group hands in.** (1) This notebook, after **Restart & Run All**, with the C1 table,
the answers and the synthesis filled in. (2) The folder `outputs/w06/coauthorship/` with the CSV
tables, the two figures and `statistics.json`. (3) A commit with a message that says what you
changed.

## 10 Optional extension: repeated ties only

*Only if you have finished.* Keep only edges with weight ≥ 2 (pairs that share at least two
publications). This is **another scenario**: it never replaces the main network. The node set
stays exactly the same; a researcher who loses all edges becomes an isolate, not a missing row.

**In symbols.** With a threshold $\tau$ (read "tau"), the scenario keeps all researchers and only
the edges whose weight reaches it:

$$V_\tau = V, \qquad E_\tau = \{\, (i, j) \in E : w_{ij} \geq \tau \,\}, \qquad \tau = 2$$

This is the shape of a **robustness contrast**: same data, one declared choice changed, every
other choice fixed, both results reported side by side. In M4 your group needs one contrast that
is justified *for its own claim*. Reusing this one only counts if you argue why a threshold on
edge weight matters for your question.

**Exercise 10.** Build `G2` with the same nodes as `G` and only the edges whose weight is at
least `SENSITIVITY_MIN_EDGE_WEIGHT`.

In [ ]:
G2 = nx.Graph()
G2.add_nodes_from(researchers)
G2.add_weighted_edges_from(e for e in edges.itertuples(index=False) if e.weight >= SENSITIVITY_MIN_EDGE_WEIGHT)
comparison = pd.DataFrame({
    "min weight 1": [G.number_of_nodes(), G.number_of_edges(), nx.number_connected_components(G), nx.number_of_isolates(G)],
    f"min weight {SENSITIVITY_MIN_EDGE_WEIGHT}": [G2.number_of_nodes(), G2.number_of_edges(),
                                                  nx.number_connected_components(G2), nx.number_of_isolates(G2)],
}, index=["researchers", "edges", "components", "isolates"])
comparison

In [ ]:
# Checkpoint — a threshold removes edges, never researchers.
assert G2.number_of_nodes() == G.number_of_nodes(), "Keep every researcher: add the nodes before the edges."
assert G2.number_of_edges() <= G.number_of_edges() and nx.number_of_isolates(G2) >= nx.number_of_isolates(G)
assert min((d["weight"] for _, _, d in G2.edges(data=True)), default=SENSITIVITY_MIN_EDGE_WEIGHT) >= SENSITIVITY_MIN_EDGE_WEIGHT

In [ ]:
(OUTPUT_DIR / "coauthorship_min_weight_2").mkdir(exist_ok=True)
comparison.to_csv(OUTPUT_DIR / "coauthorship_min_weight_2" / "sensitivity_min_edge_weight_2.csv")

**Your answer.** What changed in the connectivity, and why is this a consequence of a decision
about the network rather than a discovery about the researchers?

*Write here.*

**Teacher note.** With weight ≥ 2: 212 edges (from 364), 54 components (from 33), 51 isolates
(from 29), largest component 129 (from 148); the between-group weight share moves from 0.3216 to
0.3114. The network fragments because we decided to ignore single shared publications. A
threshold is not "removing irrelevant collaboration", and it must not be chosen to produce a
desired story.

## After class — home extension (about 1.5 h per student)

The notebook, its outputs and the synthesis were handed in during class. At home you **apply**
what was shown today. You do not start topics or classification: they have their own sessions.

1. **Research Plan v0** in `docs/research_plan_v0.md`: question, units of analysis, data,
   strategy and risks, following the Research Plan guide. Use today's audit for the units and
   the universe.
2. **Provisional variant.** In the Research Plan, name the variant your group prefers and the unit and
   denominator it will need. Do not run it.
3. **Individual run.** Each member runs this notebook once with Restart Kernel → Run All and
   records it in `docs/contributions.csv`.
4. **Evidence entry E06** in `docs/evidence_register.md` (see the box before section 9). Copy the
   synthesis to `reports/W06S01_preliminary_discussion_180_250.md`.
5. **Optional, W05 «From repeated code to one function».** Section 4 reads both tables with the
   same options (`dtype=str`, `keep_default_na=False`, `na_values=[""]`), and W07 will read them
   again. The shared `read_table` function now holds that rule in `src/icd_project/data.py`; import it here and run
   Restart Kernel → Run All: checkpoint C1 must pass unchanged. That is the W05 check that a change
   moved the code without changing what it computes.

Finish with a commit whose message says what changed and why.